In [1]:
import pandas as pd
import os
from src.data_preparation import preprocessing_anime, train_test_processing
from src.models import Baseline, ContentBased, Collaborative, Hybrid
from src.evaluate import evaluate_recommender, get_best_alpha


## Предобработка данных

In [2]:
# Загрузка исходных данных
anime_df, rating_df = pd.read_csv('anime_dataset/anime.csv'), pd.read_csv('anime_dataset/rating.csv')
# Сохранение обработанных данных об аниме в отдельном датафрейме
full_anime_df = preprocessing_anime(anime_df)
# Разбиение данных на train и test
train, test = train_test_processing(anime_df, rating_df, rating_threshold=7, random_state=42)

## Инициализация и обучение моделей

In [3]:
# Базовая модель
baseline = Baseline()
baseline.fit(full_anime_df)

In [4]:
# Контентно-ориентированная модель
content = ContentBased()
content.fit(full_anime_df)

In [5]:
# Коллаборативная модель
collab = Collaborative()
collab.fit(full_anime_df, train)

In [6]:
# Гибридная модель
hybrid = Hybrid()
hybrid.fit(full_anime_df, train)

### Нахождение оптимальных параметров гибридной модели

In [7]:
# Нахождение оптимального параметра alpha
alphas_df = get_best_alpha(full_anime_df, test, train, hybrid)
alphas_df

Валидация Hybrid_system_1.0: 100%|██████████| 500/500 [00:29<00:00, 16.75it/s]


,Модель,Hit Rate@5,Precision@5,Recall@5,MAP@5,NDCG@5
0,Hybrid_system_0.0,0.176,0.0392,0.0196,0.021540,0.042375
1,Hybrid_system_0.1,0.220,0.0524,0.0262,0.031740,0.058414
2,Hybrid_system_0.2,0.270,0.0652,0.0326,0.040373,0.072917
3,Hybrid_system_0.3,0.308,0.0780,0.0390,0.047700,0.085468
4,Hybrid_system_0.4,0.338,0.0884,0.0442,0.054067,0.096208
5,Hybrid_system_0.5,0.342,0.0856,0.0428,0.052460,0.094660
6,Hybrid_system_0.6,0.326,0.0792,0.0396,0.049080,0.089091
7,Hybrid_system_0.7,0.266,0.0660,0.0330,0.040107,0.073189
8,Hybrid_system_0.8,0.196,0.0476,0.0238,0.029567,0.053673
9,Hybrid_system_0.9,0.152,0.0368,0.0184,0.022047,0.040092


Лучшие метрики гибридная модель показывает при alpha равному 0.5. Стоит отметить, что при alpha равному 0 и 1 (то есть учет предсказаний только контно-ориентированной или коллаборативной составляющих модели) модель показывает худшие метрики, а следовательно эти два подхода взаимодополняют друг друга

## Примеры работы моделей

### Базовая модель

In [15]:
baseline.recommend(train, user_id=1000, n=10, only_id=False)

,anime_id,name,genre,type,episodes,mean_rating_anime,members,for_tfidf
0,16498,Shingeki no Kyojin,Action Drama Fantasy Shounen Super Power,TV,25,8.54,896229,Action Drama Fantasy Shounen Super Power TV
1,5114,Fullmetal Alchemist: Brotherhood,Action Adventure Drama Fantasy Magic Military ...,TV,64,9.26,793665,Action Adventure Drama Fantasy Magic Military ...
2,6547,Angel Beats!,Action Comedy Drama School Supernatural,TV,13,8.39,717796,Action Comedy Drama School Supernatural TV
3,1575,Code Geass: Hangyaku no Lelouch,Action Mecha Military School Sci-Fi Super Power,TV,25,8.83,715151,Action Mecha Military School Sci-Fi Super Powe...
4,9253,Steins;Gate,Sci-Fi Thriller,TV,24,9.17,673572,Sci-Fi Thriller TV
5,10620,Mirai Nikki (TV),Action Mystery Psychological Shounen Supernatu...,TV,26,8.07,657190,Action Mystery Psychological Shounen Supernatu...
6,226,Elfen Lied,Action Drama Horror Psychological Romance Sein...,TV,13,7.85,623511,Action Drama Horror Psychological Romance Sein...
7,22319,Tokyo Ghoul,Action Drama Horror Mystery Psychological Sein...,TV,12,8.07,618056,Action Drama Horror Mystery Psychological Sein...
8,19815,No Game No Life,Adventure Comedy Ecchi Fantasy Game Supernatural,TV,12,8.47,602291,Adventure Comedy Ecchi Fantasy Game Supernatur...
9,121,Fullmetal Alchemist,Action Adventure Comedy Drama Fantasy Magic Mi...,TV,51,8.33,600384,Action Adventure Comedy Drama Fantasy Magic Mi...


### Контентно-ориентированная модель

In [14]:
content.recommend(train, user_id=1000, n=10, only_id=False)

,anime_id,name,genre,type,episodes,mean_rating_anime,members,for_tfidf
0,25013,Akatsuki no Yona,Action Adventure Comedy Fantasy Romance Shoujo,TV,24,8.23,216674,Action Adventure Comedy Fantasy Romance Shoujo TV
1,3298,Hatenkou Yuugi,Adventure Comedy Drama Fantasy Romance Shoujo,TV,10,7.30,25885,Adventure Comedy Drama Fantasy Romance Shoujo TV
2,2787,Shakugan no Shana II (Second),Action Drama Fantasy Romance School Supernatural,TV,24,7.79,184525,Action Drama Fantasy Romance School Supernatur...
3,355,Shakugan no Shana,Action Drama Fantasy Romance School Supernatural,TV,24,7.74,297058,Action Drama Fantasy Romance School Supernatur...
4,21995,Ao Haru Ride,Comedy Drama Romance School Shoujo Slice of Life,TV,12,7.89,227417,Comedy Drama Romance School Shoujo Slice of Li...
5,145,Kareshi Kanojo no Jijou,Comedy Drama Romance School Shoujo Slice of Life,TV,26,7.66,94165,Comedy Drama Romance School Shoujo Slice of Li...
6,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,Action Adventure Comedy Fantasy Romance,TV,13,7.88,336349,Action Adventure Comedy Fantasy Romance TV
7,232,Cardcaptor Sakura,Adventure Comedy Drama Fantasy Magic Romance S...,TV,70,8.18,181249,Adventure Comedy Drama Fantasy Magic Romance S...
8,4080,Kyou kara Maou! 3rd Series,Adventure Comedy Fantasy Romance Shoujo,TV,39,7.97,19510,Adventure Comedy Fantasy Romance Shoujo TV
9,6773,Shakugan no Shana III (Final),Action Drama Fantasy Romance Supernatural,TV,24,7.73,133010,Action Drama Fantasy Romance Supernatural TV


### Коллаборативная модель

In [18]:
collab.recommend(train, user_id=1000, n=10, only_id=False)

,anime_id,name,genre,type,episodes,mean_rating_anime,members,for_tfidf
0,32281,Kimi no Na wa.,Drama Romance School Supernatural,Movie,1,9.37,200630,Drama Romance School Supernatural Movie
1,5114,Fullmetal Alchemist: Brotherhood,Action Adventure Drama Fantasy Magic Military ...,TV,64,9.26,793665,Action Adventure Drama Fantasy Magic Military ...
2,31757,Kizumonogatari II: Nekketsu-hen,Action Mystery Supernatural Vampire,Movie,1,8.73,34347,Action Mystery Supernatural Vampire Movie
3,23273,Shigatsu wa Kimi no Uso,Drama Music Romance School Shounen,TV,22,8.92,416397,Drama Music Romance School Shounen TV
4,9253,Steins;Gate,Sci-Fi Thriller,TV,24,9.17,673572,Sci-Fi Thriller TV
5,431,Howl no Ugoku Shiro,Adventure Drama Fantasy Romance,Movie,1,8.74,333186,Adventure Drama Fantasy Romance Movie
6,2904,Code Geass: Hangyaku no Lelouch R2,Action Drama Mecha Military Sci-Fi Super Power,TV,25,8.98,572888,Action Drama Mecha Military Sci-Fi Super Power TV
7,28851,Koe no Katachi,Drama School Shounen,Movie,1,9.05,102733,Drama School Shounen Movie
8,28977,Gintama°,Action Comedy Historical Parody Samurai Sci-Fi...,TV,51,9.25,114262,Action Comedy Historical Parody Samurai Sci-Fi...
9,32935,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,Comedy Drama School Shounen Sports,TV,10,9.15,93351,Comedy Drama School Shounen Sports TV


### Гибридная модель

In [19]:
hybrid.recommend(train, user_id=1000, n=10, alpha=0.5, only_id=False)

,anime_id,name,genre,type,episodes,mean_rating_anime,members,for_tfidf
0,32281,Kimi no Na wa.,Drama Romance School Supernatural,Movie,1,9.37,200630,Drama Romance School Supernatural Movie
1,4181,Clannad: After Story,Drama Fantasy Romance Slice of Life Supernatural,TV,24,9.06,456749,Drama Fantasy Romance Slice of Life Supernatur...
2,431,Howl no Ugoku Shiro,Adventure Drama Fantasy Romance,Movie,1,8.74,333186,Adventure Drama Fantasy Romance Movie
3,25013,Akatsuki no Yona,Action Adventure Comedy Fantasy Romance Shoujo,TV,24,8.23,216674,Action Adventure Comedy Fantasy Romance Shoujo TV
4,23273,Shigatsu wa Kimi no Uso,Drama Music Romance School Shounen,TV,22,8.92,416397,Drama Music Romance School Shounen TV
5,6811,InuYasha: Kanketsu-hen,Action Adventure Comedy Demons Fantasy Magic R...,TV,26,8.37,99128,Action Adventure Comedy Demons Fantasy Magic R...
6,12365,Bakuman. 3rd Season,Comedy Drama Romance Shounen,TV,25,8.71,133620,Comedy Drama Romance Shounen TV
7,11665,Natsume Yuujinchou Shi,Drama Fantasy Shoujo Slice of Life Supernatural,TV,13,8.75,98431,Drama Fantasy Shoujo Slice of Life Supernatura...
8,10408,Hotarubi no Mori e,Drama Romance Shoujo Supernatural,Movie,1,8.61,197439,Drama Romance Shoujo Supernatural Movie
9,249,InuYasha,Action Adventure Comedy Demons Fantasy Magic R...,TV,167,7.89,281632,Action Adventure Comedy Demons Fantasy Magic R...


## Оценка метрик моделей

In [12]:
results_df = pd.DataFrame()
results_df = evaluate_recommender(full_anime_df, test, train, baseline, 'Baseline_system', results_df, k=5)
results_df = evaluate_recommender(full_anime_df, test, train, content, 'Content_system', results_df, k=5)
results_df = evaluate_recommender(full_anime_df, test, train, collab, 'Collaborative_system', results_df, k=5)
results_df = evaluate_recommender(full_anime_df, test, train, hybrid, 'Hybrid_system', results_df, k=5, alpha=0.5)
results_df

Валидация Hybrid_system: 100%|██████████| 49486/49486 [49:43<00:00, 16.59it/s]  


,Модель,Hit Rate@5,Precision@5,Recall@5,MAP@5,NDCG@5
0,Baseline_system,0.434143,0.118163,0.059081,0.075862,0.131638
1,Content_system,0.124116,0.029544,0.014772,0.016766,0.031430
2,Collaborative_system,0.151134,0.033828,0.016914,0.017583,0.035151
3,Hybrid_system,0.298408,0.073609,0.036804,0.044644,0.081313


Лучшие метрики показывает базовая модель. Она предлагает 57% пользователей как минимум одно релевантное аниме, в целом рекомендует больше релевантных объектов и лучше ранжирует их. Вероятно, это связано с тем, что пользователи предпочитают для просмотра высокооцененные тайтлы, за счет чего они становятся самыми популярными. Особенно об этом свидетельствует превосходство модели по метрикам, учитывающим ранжирование

Стоит также обратить внимание, что гибридная модель показывает более высокие метрики, в сравнении с контентно-ориентированной и коллаборативной моделями

In [13]:
# Сохранение результатов
os.makedirs('results', exist_ok=True)
results_df.to_csv('results/metrics.csv', index=False)
results_df.to_markdown('results/metrics.md', index=False)